# 3차시 실습 — PRD & MVP 가설

**무엇을·누구를·왜 — 방향을 한 장에 고정하고, 가설로 검증한다**

## 🧪 이 노트북 사용법

- 이 노트북은 워크샵 **실습(Codex CLI)의 실행 가능한 동반 자료**입니다. 각자 발급받은 **`OPENAI_API_KEY`** 로 바로 돌아갑니다.
- 실제 캠프에서는 **터미널의 Codex CLI**로 같은 작업을 합니다. 그래서 각 단계마다
  - **▶︎ 노트북에서 실행** (파이썬 셀) 과
  - **⌨️ 터미널 Codex 명령** (복붙용) 을 함께 적어 두었습니다.
- 위에서부터 **순서대로** 셀을 실행하세요 (`Shift+Enter`).
- 황금률: **작게 요청 → 확인 → 수정** (1차시 iteration loop).

> ⚠️ API 호출에는 약간의 토큰 비용이 듭니다. 기본 모델은 저렴한 `gpt-4o-mini`. **설치·키 발급·Codex 로그인은 0차시에서 끝낸 전제**입니다 — 안 됐으면 `0_setup/설치_가이드.md` 를 먼저 보세요.

In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 실행
%pip install -q openai
import os, getpass, re, pathlib, json
from openai import OpenAI
from IPython.display import IFrame, Markdown, display

# 각자 발급받은 OPENAI_API_KEY 입력 (입력값은 화면에 보이지 않습니다)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY를 붙여넣고 Enter: ")

client = OpenAI()
MODEL = "gpt-4o-mini"   # 품질을 더 원하면 "gpt-4o" 또는 "gpt-4.1" 로 바꾸세요

def ask(prompt, system=None, **kw):
    """프롬프트 하나를 보내고 답변 텍스트를 돌려준다."""
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, messages=msgs, **kw)
    return r.choices[0].message.content

def show(text):
    display(Markdown(text))

print("준비 완료! MODEL =", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료! MODEL = gpt-4o-mini


## PRD 4요소 + 비목표 

**① 문제 정의  ② 타깃 사용자(유저 스토리)  ③ 핵심 기능(1~3, wow)  ④ 성공 기준(수용 기준)  + 가드레일: 비목표**

아래 변수를 **여러분의 아이디어로** 바꾼 뒤 실행하면, Codex가 한 장 PRD 초안을 만들어 줍니다.
(예시는 1차시 점심 추천기로 채워져 있습니다.)

> 프로 팁 (S63~S65): 에이전트가 읽을 문서일수록 **구조화**(불릿·표)하고, **모호어**("빠르게"→"3초 안에")를 없애고, **비목표**로 범위를 막습니다. — 출처: Cagan(SVPG)·ChatPRD, 슬라이드 부록 참고

In [2]:
idea = {
    "문제": "혼밥 직장인이 매일 점심 메뉴 고르기에 피로를 느낀다",
    "타깃": "점심을 혼자 빠르게 해결하는 20~30대 직장인",
    "유저스토리": "혼밥 직장인으로서 3초 안에 메뉴를 정하고 싶다, 고민할 시간이 아깝기 때문에",
    "핵심기능": "버튼 한 번에 랜덤 추천 + 음식 종류 필터 (오늘 데모할 1개 flow, wow: 3초 안에 결정 끝)",
    "비목표": "회원가입, 지도/맛집 연동, 결제",
    "성공기준": "버튼 클릭 → 1초 안에 추천 표시 / 데모에서 5명 중 3명이 '다시 쓰겠다'고 답한다",
}

prompt = f"""다음 입력으로 한 장짜리 PRD를 작성해줘.
형식:
# [제품명] PRD
- 문제:
- 타깃: (유저 스토리 1줄)
- 핵심 기능(1~3): (wow 명시)
- 비목표(안 만드는 것):
- 성공 기준: (측정 가능한 수용 기준)

입력:
- 문제: {idea['문제']}
- 타깃: {idea['타깃']}
- 유저 스토리: {idea['유저스토리']}
- 핵심기능: {idea['핵심기능']}
- 비목표: {idea['비목표']}
- 성공기준: {idea['성공기준']}
간결하게, 한국어로. 모호한 표현("빠르게", "편하게") 대신 측정 가능한 표현을 써."""

prd = ask(prompt)
show(prd)
pathlib.Path("team_prd.md").write_text(prd, encoding="utf-8")
print("\nteam_prd.md 저장 — 5차시 팀 워크샵에서 이어서 씁니다.")

# [점심 추천 서비스] PRD
- 문제: 혼밥 직장인이 매일 점심 메뉴 고르기에 피로를 느낀다.
- 타깃: 점심을 혼자 해결하는 20~30대 직장인
- 유저 스토리: 혼밥 직장인으로서 3초 안에 메뉴를 정하고 싶다, 고민할 시간이 아깝기 때문에.
- 핵심 기능: 버튼 한 번에 랜덤 추천 + 음식 종류 필터 (오늘 데모할 1개 flow, wow: 3초 안에 결정 끝)
- 비목표: 회원가입, 지도/맛집 연동, 결제
- 성공 기준: 버튼 클릭 → 1초 안에 추천 표시 / 데모에서 5명 중 3명이 '다시 쓰겠다'고 답한다.


team_prd.md 저장 — 4·5차시에서 이어서 씁니다.


## PRD 검토 — AI를 리뷰어로 

AI 초안을 **그대로 쓰지 않습니다.** 6항목 기준(4요소 + 비목표 + 모호어)으로 스스로 점검하게 시켜 약한 곳을 찾습니다.

In [3]:
check_prompt = f"""아래 PRD를 다음 6가지로 점검해줘. 각 항목을 통과/미흡으로 표시하고, 미흡하면 한 줄 개선안을 줘.
1) 문제가 한 문장으로 구체적인가
2) 타깃이 '모두'가 아니라 구체적인가 (유저 스토리 형식인가)
3) 핵심 기능이 1개 flow로 좁혀졌는가 (wow가 보이는가)
4) 비목표(안 만드는 것)가 명시됐는가
5) 성공 기준이 측정 가능한 '수용 기준'인가
6) 모호어("빠르게", "편하게", "예쁘게" 등)가 남아있지 않은가

PRD:
{prd}"""
show(ask(check_prompt))

1) 문제가 한 문장으로 구체적인가: **통과**
   
2) 타깃이 '모두'가 아니라 구체적인가 (유저 스토리 형식인가): **통과**

3) 핵심 기능이 1개 flow로 좁혀졌는가 (wow가 보이는가): **통과**

4) 비목표(안 만드는 것)가 명시됐는가: **통과**

5) 성공 기준이 측정 가능한 '수용 기준'인가: **미흡**  
   개선안: 성공 기준을 더욱 구체화하여, "버튼 클릭 후 1초 이내에 80% 이상의 사용자에게 추천이 정확하다고 응답받는다"와 같이 설정하는 것이 좋습니다.

6) 모호어("빠르게", "편하게", "예쁘게" 등)가 남아있지 않은가: **통과**

## 미니 실습 ① — 나쁜 PRD 고치기 (모호어 사냥)

아래 `bad_prd`는 모호어투성이입니다. **`fixed_prd`를 직접 고쳐 쓴 뒤** 실행하면 AI가 채점합니다.
사람은 모호함을 상식으로 메우지만, **에이전트는 아무거나 만듭니다** — "빠르게"→"3초 안에"처럼 측정 가능하게.

In [4]:
bad_prd = """# 만능 일정 앱 PRD
- 문제: 사람들이 일정 관리가 불편하다
- 타깃: 모든 바쁜 현대인
- 핵심 기능: 일정을 편하게 관리, 알림을 빠르게, UI를 예쁘게
- 성공 기준: 사용자 반응이 좋다"""

# TODO: 아래를 직접 고쳐 보세요 (타깃 좁히기 · 모호어 → 측정 가능 · 비목표 추가)
fixed_prd = """# 만능 일정 앱 PRD
- 문제:
- 타깃: (유저 스토리 1줄)
- 핵심 기능(1~3):
- 비목표(안 만드는 것):
- 성공 기준:"""

show(ask(f"""학생이 모호한 PRD를 고쳤어. 채점해줘.
각 항목을 통과/미흡으로 표시하고, 아직 남은 모호어("편하게", "빠르게" 류)가 있으면 콕 집어줘. 격려 한 줄로 마무리.

[원본(나쁜 예)]
{bad_prd}

[학생이 고친 버전]
{fixed_prd}"""))

채점 결과:

- 문제: 미흡 (문제가 구체적으로 기술되지 않음)
- 타깃: 통과 (유저 스토리가 명확히 언급되면 더 좋음)
- 핵심 기능(1~3): 미흡 (구체적인 기능이 제시되지 않음)
- 비목표(안 만드는 것): 미흡 (비목표가 구체적으로 기재되지 않음)
- 성공 기준: 미흡 (정량적 혹은 정성적 기준이 명시되지 않음)

모호어: "편하게", "빠르게", "예쁘게" 등의 표현이 여전히 남아있음.

더 구체적이고 명확한 내용을 추가하면 더욱 완성도 높은 PRD가 될 거예요. 잘 하고 있어요!

## 미니 실습 ② — 비목표 3개 적기

**"안 만드는 것"을 적어야 에이전트의 범위 번짐(scope creep)을 막습니다.** 내 아이디어에서 이번엔 안 만들 것 3개를 적고 실행하세요 

In [5]:
non_goals = [
    "예: 회원가입",   # ← 여러분의 비목표로 교체하세요
    "예: 결제",
    "예: 푸시 알림",
]

show(ask(f"""아래 PRD와 비목표 목록을 봐줘.
1) 이 비목표가 PRD의 핵심 기능과 충돌하지 않는지
2) PRD에 더 추가하면 좋을 비목표 후보 2개 (범위 번짐이 잦은 것 위주)
를 짧게 한국어로.

PRD:
{prd}

비목표:
{non_goals}"""))

1) 비목표 목록에 있는 '회원가입'과 '결제'는 PRD의 핵심 기능과 충돌하지 않습니다. 이 기능들은 사용자 경험을 단순하게 유지하는 것과는 관련이 없으며, 추천 서비스를 제공하는 데 필요한 것이 아닙니다. '푸시 알림'도 사용자에게 추가 정보를 제공할 수 있지만, 현재 목표인 3초 내 메뉴 추천과는 관련이 없습니다.

2) 추가하면 좋을 비목표 후보:
- 사용자 맞춤형 추천 기능 (개인적인 선호를 반영하지 않음)
- 다른 사용자 피드백 기능 (사회적 상호작용 요소 추가하지 않음)

## 스펙킷(Spec Kit) — 명세 중심 워크플로 

**PRD(무엇을) → 스펙(어떻게 동작) → 구현 → 검증**, 검증에서 스펙으로 되돌아오는 반복.
스펙킷 = 스펙(명세)을 *단일 진실*로 삼아 구현·검증을 정렬하는 **도구**.

## A.Spec Kit 사용법 — 터미널 + Codex

> 스펙킷은 코딩 에이전트(Codex)와 함께 쓰는 **CLI 도구**라, 노트북이 아니라 **터미널**에서 진행합니다.

### ① 설치 & 초기화 (uv 필요)
```bash
# 매번 받아 실행 (one-off)
uvx --from git+https://github.com/github/spec-kit.git specify init my-app --integration codex

# 또는 영구 설치 후 사용
uv tool install specify-cli --from git+https://github.com/github/spec-kit.git
specify init my-app --integration codex

specify self check          # 설치·버전 확인
specify integration list    # 지원 에이전트 목록
```
- `--integration codex` → **Codex CLI**용 명령 파일·컨텍스트 규칙·디렉터리를 자동 생성합니다.

### ② Codex 안에서 실습 — `$speckit-*` 8개 명령 전부 돌려보기

`my-app` 폴더로 들어가 **`codex`** 를 실행한 뒤, 대화창에 아래 명령을 **순서대로** 입력합니다.
Codex는 **스킬(skills) 모드**라 모든 명령 앞에 **`$`** 를 붙입니다.

```text
cd my-app
codex
```

**`$speckit-*` 전체 명령 (8개)** — 프로덕션 권장 순서:

| 순서 | 명령 | 하는 일 | 산출물(예) |
|---|---|---|---|
| 1 | `$speckit-constitution` | 프로젝트 원칙·개발 규칙 정의 | `memory/constitution.md` |
| 2 | `$speckit-specify` | **무엇을·왜** (요구사항·유저스토리) | `specs/.../spec.md` |
| 3 | `$speckit-clarify` | 모호한 부분을 Codex가 되물어 보완 | spec.md 갱신 |
| 4 | `$speckit-checklist` | 요구사항 품질 체크리스트 생성·검증 | `checklist.md` |
| 5 | `$speckit-plan` | 기술 설계 (스택·구조 선택) | `plan.md` |
| 6 | `$speckit-tasks` | 작업 분해 (구현 단위) | `tasks.md` |
| 7 | `$speckit-analyze` | 스펙↔계획↔작업 정합성·커버리지 점검 | 분석 리포트 |
| 8 | `$speckit-implement` | 계획대로 실제 구현 실행 | 코드 파일 |

> 빠른 실험만 할 땐 1·3·4·7을 건너뛰고 **specify → plan → tasks → implement** 4개만 써도 됩니다. 이번 실습은 **8개를 모두** 경험합니다.

### ③ 워크드 예시 — 「오늘 뭐 먹지?」 메뉴 추천 + 맛집 길안내 앱

8개 명령으로 **하나의 앱**을 끝까지 만들어 봅니다. 만들 앱:

> **오늘 뭐 먹지?** — 기분·예산·카테고리로 **메뉴를 추천**하고, 그 메뉴를 파는 **근처 맛집**을 거리·평점순으로 보여준 뒤 **지도 길안내**로 연결.

기능(핵심 + 확장):
- 🍜 **메뉴 추천**: 카테고리(한/중/일/양/분식/아무거나)·기분·예산·인원 → 추천 1개 + 대안 2개, **다시 추천(흔들기)**
- 📍 **근처 맛집**: 추천 메뉴 기준 맛집을 **거리·평점·가격대·영업중 여부**로 정렬/필터
- 🧭 **길안내**: 선택한 맛집을 지도 앱(카카오맵/구글맵)으로 바로 열기
- 🔁 **최근 회피**: 방금 먹은 메뉴는 잠시 추천에서 제외
- ⭐ **찜 & 방문기록**: localStorage 저장
- 👥 (확장) **그룹 투표**, 🕒 (확장) **"지금 영업중만"** 필터

아래 각 명령을 **순서대로** Codex 대화창에 입력하세요. (명령은 짧게, 상세는 다음 메시지로)

#### 1·2 — 원칙과 스펙

```text
$speckit-constitution
이 프로젝트의 원칙:
- 사용자는 3번의 입력 안에 "뭘 먹을지 + 어디 갈지"를 정할 수 있어야 한다(속도 최우선)
- 모바일 한 손 사용 우선, 버튼 44px 이상
- 외부 API 키 없이도 데모 데이터로 동작해야 한다(키는 선택적 확장)
- 접근성: 색만으로 정보 전달 금지
- 단일 HTML 파일, 한국어 주석으로 단순하게
```

```text
$speckit-specify 기분·예산·카테고리로 메뉴를 추천하고, 그 메뉴를 파는 근처 맛집을 거리·평점순으로 보여준 뒤 지도 길안내로 연결하는 웹앱.
```
↳ 이어서 **다음 메시지**로 상세 붙이기:
```text
유저스토리:
- 카테고리·기분·예산·인원을 고르면 메뉴 1개 + 대안 2개를 추천받는다
- "다시 추천"으로 즉시 다른 메뉴를 받는다
- 추천 메뉴를 파는 근처 맛집을 거리/평점/가격대/영업중 여부로 본다
- 맛집을 고르면 지도 앱 길안내로 연결된다
- 방금 먹은 메뉴는 잠시 추천에서 제외한다(최근 회피)
- 찜한 맛집과 방문 기록을 저장한다
이번 범위 밖: 회원가입, 결제, 실시간 예약
```
> 산출물: `memory/constitution.md`, `specs/001-.../spec.md`

#### 3·4 — 모호점 보완 & 체크리스트

```text
$speckit-clarify
```
Codex가 되묻는 질문에 답하는 예:
```text
- 맛집 데이터 출처? → 데모용 내장 JSON 20개로 시작, 추후 카카오 로컬 API 선택 연결
- "근처" 거리 기준? → 기본 2km, 슬라이더로 0.5~5km
- 평점 출처? → 데모 데이터의 고정 평점(1~5), 실제 연동은 확장
- 영업중 판단? → 데이터의 영업시간 + 현재 시각 비교
```

```text
$speckit-checklist
```
> 생성되는 체크 항목(예): 빈 입력 처리 · 필터가 과해 결과 0개일 때 · 위치 권한 거부 시 폴백 · 영업종료 맛집 표시 방식 · 색 대비/접근성 · localStorage 용량
> 산출물: spec.md 갱신, `checklist.md`

#### 5·6 — 기술 설계 & 작업 분해

```text
$speckit-plan 프레임워크 없이 단일 index.html + 내장 CSS/JS. 상태·찜·방문기록은 localStorage. 맛집은 내장 demoRestaurants(JSON). 길안내는 지도 검색 URL(카카오맵/구글맵)을 새 탭으로 연다. 위치는 navigator.geolocation, 거부 시 도시 선택 폴백. 거리는 Haversine로 계산.
```
> 설계 산출(예): 데이터 모델 `restaurant{ name, category, lat, lng, rating, price, hours }` · 화면 3개(입력 → 추천 → 맛집/길안내) · 추천 알고리즘(가중 랜덤 + 최근 회피)

```text
$speckit-tasks
```
> 작업 분해(예): ①3화면 레이아웃 ②입력폼+검증 ③추천엔진+다시추천 ④demoRestaurants+거리계산 ⑤맛집 정렬/필터(영업중·평점·거리) ⑥길안내 링크 ⑦찜/방문기록 localStorage ⑧빈 상태·접근성 처리
> 산출물: `plan.md`, `tasks.md`

#### 7·8 — 정합성 점검 & 구현

```text
$speckit-analyze
```
> 점검 결과(예): spec엔 **"최근 회피"** 가 있는데 tasks에 빠짐 → 작업 추가 / plan의 **위치 거부 폴백**이 tasks에 누락 → 보강. **여기서 먼저 고친 뒤** 구현으로.

```text
$speckit-implement
```
> `index.html` 생성 → 브라우저로 열어 확인 → 스펙과 다르면 spec/plan 수정 후 **다시 `$speckit-implement`** (검증·반복).

**확인 포인트**
- 카테고리/예산 고르고 추천 → 메뉴 1 + 대안 2가 뜨는가, "다시 추천"이 동작하는가
- 추천 메뉴의 근처 맛집이 거리·평점순으로 뜨고 "지금 영업중만" 필터가 되는가
- 맛집의 길안내 버튼이 지도 앱(새 탭)으로 열리는가
- 찜/방문기록이 새로고침 후에도 남는가

> ⚠️ **자주 막히는 점**
> - 명령은 반드시 **`$` 로 시작** (예: `$speckit-specify`).
> - 긴 설명을 한 번에 붙여넣으면 `[Pasted Content 1234 chars]` 로 뭉개짐 → 명령 줄엔 **짧게**, 상세는 **다음 메시지**로.
> - 명령·플래그는 자주 바뀝니다. `specify --help` 와 **github.com/github/spec-kit** 에서 최신 확인.

### ④ 이제 **내 PRD로** — 스펙킷 입력 자동 생성

위 「오늘 뭐 먹지?」는 **보기**였습니다. 이제 앞에서 만든 **여러분의 PRD(`prd` / `team_prd.md`)** 를 그대로 `$speckit-*` 입력으로 바꿔서 Codex에 붙여넣습니다.

아래 셀을 실행하면, **내 PRD 기준**으로 1~5단계에 붙여넣을 명령 입력이 만들어집니다. (`speckit_inputs.md` 로 저장)

In [6]:
speckit_inputs = ask(f"""아래 PRD를 GitHub Spec Kit + Codex 입력으로 변환해줘.
바로 복사해 Codex 대화창에 붙여넣을 수 있게, 아래 형식 그대로 한국어로 출력해.

== 1) $speckit-constitution 뒤에 붙일 원칙 (5줄, 속도·모바일·접근성·단순함 관점) ==
== 2) $speckit-specify 뒤에 붙일 '짧은 한 줄' 요약 ==
== 3) (다음 메시지) 유저스토리 4~6개 + '이번 범위 밖' 2~3개 ==
== 4) $speckit-clarify 에서 Codex가 물을 법한 질문 3개와 '권장 답' ==
== 5) $speckit-plan 뒤에 붙일 기술 스택 한 줄 (프레임워크 없이 단일 HTML + 내장 JS, localStorage 기준) ==

PRD:
{prd}""")
show(speckit_inputs)
pathlib.Path("speckit_inputs.md").write_text(speckit_inputs, encoding="utf-8")
print("\nspeckit_inputs.md 저장 — Codex 대화창에 1)부터 순서대로 붙여넣으세요.")
print("순서: $speckit-constitution → specify → clarify → checklist → plan → tasks → analyze → implement")

== 1) $speckit-constitution 뒤에 붙일 원칙 (5줄, 속도·모바일·접근성·단순함 관점) ==
- 서비스는 사용자에게 빠른 결정 지원을 제공해야 하며, 3초 이내에 추천이 이루어져야 한다.
- 모바일 환경에서의 사용성을 최우선으로 고려하여 디자인해야 한다.
- 다양한 음식 종류 필터를 통해 접근성을 높이고, 개인의 선호도를 반영할 수 있어야 한다.
- 서비스는 단순한 인터페이스를 통해 사용자가 쉽게 이해하고 사용할 수 있도록 구축해야 한다.
- 불필요한 기능은 배제하여 혼밥 직장인이 스트레스 없이 메뉴를 선택할 수 있도록 해야 한다.

== 2) $speckit-specify 뒤에 붙일 '짧은 한 줄' 요약 ==
혼밥 직장인을 위한 버튼 클릭으로 3초 안에 랜덤 점심 메뉴 추천 서비스.

== 3) (다음 메시지) 유저스토리 4~6개 + '이번 범위 밖' 2~3개 ==
- 유저 스토리:
  1. 나는 혼밥 직장인으로서 버튼을 클릭하면 즉시 랜덤 메뉴를 추천받고 싶다.
  2. 나는 음식 종류를 필터링하여 내가 좋아하는 요리만 추천받고 싶다.
  3. 나는 빠르게 메뉴를 선택하고 싶어서 3초 내에 추천 내용을 보고 결정하고 싶다.
  4. 나는 추천받은 메뉴를 친구와 공유하고 싶다.
  
- 이번 범위 밖:
  1. 나는 회원가입 없이 서비스에 접근할 수 있어야 한다.
  2. 나는 주변 맛집과의 연동 기능이 필요하지 않다.
  3. 나는 결제 기능이 필요 없는 간단한 서비스를 원한다.

== 4) $speckit-clarify 에서 Codex가 물을 법한 질문 3개와 '권장 답' ==
1. 버튼 클릭 후 추천 메뉴는 어떻게 제공되나요?
   - 웹 페이지에서 랜덤 알고리즘을 통해 즉각적으로 추천 메뉴가 표시됩니다.
   
2. 필터링 기능은 어떤 방식으로 작동하나요?
   - 사용자가 선호하는 음식 종류를 체크하면 해당 종류에 맞는 추천만 제공합니다.
   
3. 추천 메뉴는 어떤 근거로 선정되나요?
   - 추천 내용은 데이터베이스에 저장된 다양한 메뉴 목록에서 랜덤으로 선택됩니다.

== 5) $speckit-plan 뒤에 붙일 기술 스택 한 줄 (프레임워크 없이 단일 HTML + 내장 JS, localStorage 기준) ==
단일 HTML 파일과 내장 JavaScript를 사용하여 localStorage에 추천 메뉴 데이터를 저장하고 관리합니다.


speckit_inputs.md 저장 — Codex 대화창에 1)부터 순서대로 붙여넣으세요.
순서: $speckit-constitution → specify → clarify → checklist → plan → tasks → analyze → implement


> 흐름 요약: **PRD(앞 단계) → `$speckit-specify`의 입력**으로 직결됩니다.
> 즉 PRD가 스펙킷의 출발점(무엇을·왜)이고, 스펙킷이 그걸 받아 **스펙→계획→작업→구현**으로 전개합니다.
> 4차시에서는 이 앱을 저장(localStorage)하고 GitHub Pages로 공개합니다.

## MVP 가설 사고

MVP = 가설을 검증할 **최소 기능 제품**. "사용자는 X를 원할 것이다"를 검증 대상으로 명시합니다.
검증 루프: **가설 → MVP → 데모 → 학습 → (수정)**.

In [7]:
hyp = ask(f"""아래 PRD를 바탕으로:
1) 검증할 핵심 '가설' 1문장 ("사용자는 ~할 것이다")
2) 그 가설을 검증할 최소 MVP의 wow 기능 1개
3) 데모에서 관찰할 지표 1개
를 한국어 불릿으로.

PRD:
{prd}""")
show(hyp)

- 사용자는 버튼 클릭 한 번으로 3초 이내에 점심 메뉴를 추천 받을 것이다.
- 최소 MVP의 wow 기능: 버튼 클릭 시 1초 이내에 랜덤 점심 메뉴 추천 기능.
- 데모에서 관찰할 지표: 5명 중 '다시 쓰겠다'고 응답한 사용자 비율. (목표: 3명 이상)

## 미니 실습 ③ — 가정 3개 적기

Cagan Step 6: **"가정을 적고, 의심하라."** 적지 않은 가정은 검증도 못 합니다.
내 아이디어가 성공하려면 **참이어야 하는 가정 3개**를 적고 실행하세요

In [8]:
assumptions = [
    "예: 직장인은 점심 메뉴를 3초 안에 정하고 싶어한다",   # ← 여러분의 가정으로 교체하세요
    "예: 랜덤 추천이어도 필터만 있으면 만족한다",
    "예: 회원가입 없이도 다시 방문한다",
]

show(ask(f"""아래 PRD와 가정 목록이야. 각 가정에 대해:
1) 이번 데모에서 검증 가능한가 (가능/불가)
2) 가장 싸게 검증하는 방법 한 줄
을 표로 정리해줘. 마지막에 '가장 위험한 가정' 1개를 골라 이유 한 줄을 붙여줘.

PRD:
{prd}

가정:
{assumptions}"""))

| 가정                                      | 검증 가능 여부 | 검증 방법                                   |
|-----------------------------------------|---------------|----------------------------------------|
| 직장인은 점심 메뉴를 3초 안에 정하고 싶어한다     | 가능          | 간단한 설문조사를 통해 직장인의 의견을 수집한다.           |
| 랜덤 추천이어도 필터만 있으면 만족한다            | 가능          | 필터 기능이 있는 랜덤 추천 시연 후 만족도 설문조사 실시.      |
| 회원가입 없이도 다시 방문한다                  | 불가          | 현재 진행 중인 데모에서는 장기적인 방문 데이터를 수집할 수 없다. |

**가장 위험한 가정:** '회원가입 없이도 다시 방문한다'  
이유: 만약 사용자가 다시 방문하지 않는다면, 서비스의 지속 가능성이 위협받을 수 있다.

## ✅ 마무리 체크리스트

- [ ] 내 아이디어로 PRD(4요소 + 비목표)를 채웠다
- [ ] PRD 초안을 생성했다(team_prd.md)
- [ ] 6항목 기준으로 검토·개선했다(모호어 사냥 포함)
- [ ] 나쁜 PRD를 측정 가능하게 고쳐봤다(미니 실습 ①)
- [ ] 비목표 3개를 적었다(미니 실습 ②)
- [ ] 정식 스펙킷 `$speckit-*` 8개 명령을 체험했다(A파트)
- [ ] 검증할 가설 1문장 + 가정 3개를 적었다(미니 실습 ③)